In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler

from pymoo.core.problem import Problem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize
from pymoo.operators.sampling.lhs import LHS
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.indicators.hv import HV

In [ ]:
# start wall-clock timer for the whole notebook so total runtime is reported at the end
notebook_start = time.time()

## Global configuration of the NN model

In [ ]:
# Problem constants
CSV_FILE = '6 - MaintenancePlate_TrainingData.csv' # FEA dataset
INPUT_COLS = ['W1_m', 'W2_m', 'R_m', 't_m']  # 4 design variables
STRESS_COL = 'MaxVonMisesStress_MPa' # target: max von Mises stress from FEA
LOWER_BOUNDS = np.array([0.3, 0.1, 0.03, 0.01]) # design variable lower bounds
UPPER_BOUNDS = np.array([0.7, 0.15, 0.07, 0.02]) # design variable upper bounds
RHO = 2700.0 # aluminium alloy density

# NN defaults (they progressively get wiped out as the parameters are optimised)
N_HIDDEN_DEFAULT = 1 # starting hidden layers
N_NEURONS_DEFAULT = 16 # starting neurons per layer
ACTIVATION_DEFAULT = 'relu' # starting activation function
LR_DEFAULT = 0.001  # starting learning rate
BATCH_DEFAULT = 16 # starting batch size
PATIENCE_DEFAULT = 25 # starting early stopping patience
SEED = 1 # stays the same throughout optimisation
THRESHOLD = 1.15 # tolerance band used to pick fastest model within 15% of best RMSE

# Read the FEA dataset, trims it down to the first n rows
def load_data(n_samples):
    df = pd.read_csv(CSV_FILE).iloc[:n_samples]  # Return inputs and objective as NumPy arrays
    return df[INPUT_COLS].values, df[STRESS_COL].values


# Builds a sequential dense network with n_hidden layers of n_neurons each
def build_nn_model(input_dim, n_hidden, n_neurons, activation, lr):
    model = keras.Sequential([layers.Input(shape=(input_dim,))] + [layers.Dense(n_neurons, activation=activation) for _ in range(n_hidden)] + [layers.Dense(1)])  # single output: predicted stress)
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr), loss='mse', metrics=['RootMeanSquaredError'])
    return model


# Trains the NN once with the given hyperparameters and returns metrics
def train_and_evaluate_nn(n_samples, n_hidden, n_neurons, activation, lr, batch_size, patience, seed=SEED):
    X, y = load_data(n_samples)  # loads data
    # 70/15/15 split fixed seed for reproducibility
    X_tr, X_temp, y_tr, y_temp = train_test_split(X, y, test_size=0.30, random_state=seed, shuffle=True)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=seed, shuffle=True)

    # Scale inputs and outputs
    xs = MinMaxScaler()   # bounded design space maps to [0, 1]
    ys = StandardScaler() # standardise y
    X_tr_s = xs.fit_transform(X_tr) # fit + transform on training inputs
    X_val_s = xs.transform(X_val) # transform only
    X_test_s = xs.transform(X_test) # transform only
    y_tr_s = ys.fit_transform(y_tr.reshape(-1, 1)).ravel()
    y_val_s = ys.transform(y_val.reshape(-1, 1)).ravel()

    t0 = time.time()  # calculates the training time only (fitting)

    tf.keras.backend.clear_session()  # wipe Keras state so each run starts fresh
    keras.utils.set_random_seed(seed)  # fixed seed for reproducibility
    model = build_nn_model(X.shape[1], n_hidden, n_neurons, activation, lr) # new NN
    cb = keras.callbacks.EarlyStopping(monitor='val_loss', patience=patience, restore_best_weights=True) # stops training when val_loss stalls, rolls back to best epoch
    history = model.fit(X_tr_s, y_tr_s, validation_data=(X_val_s, y_val_s), epochs=1000, batch_size=batch_size, verbose=0, callbacks=[cb])
    y_pred_s = model.predict(X_test_s, verbose=0).ravel() # predict on held-out test set
    y_pred = ys.inverse_transform(y_pred_s.reshape(-1, 1)).ravel()  # back to MPa
    rmse = np.sqrt(mean_squared_error(y_test, y_pred)) # RMSE calculation
    r2 = r2_score(y_test, y_pred) # R^2
    n_epochs = len(history.history['loss']) # how many epochs early stopping let it run

    train_time = time.time() - t0

    # Return everything downstream
    return (model, xs, ys, X_tr, X_val, X_test, y_tr, y_val, y_test, y_pred, n_epochs, float(rmse), float(r2), train_time)


# Wraps the multi-objective problem (mass + stress) and runs NSGA-II
def run_nsga2(nn_model, xs, ys, pop_size, n_gen):
    class PlateProblem(Problem):

        def __init__(self):
            super().__init__(n_var=4, n_obj=2, xl=LOWER_BOUNDS, xu=UPPER_BOUNDS)  # 4 design variables w their bounds and 2 objectives

        def _evaluate(self, X, out, *args, **kwargs):
            W1, W2, R, t = X[:, 0], X[:, 1], X[:, 2], X[:, 3]  # unpack the four design variables
            mass   = RHO * t * (4*W1**2 - 4*W2**2 + (4 - np.pi)*R**2)  # analytical mass formula (Eq. 1)
            stress = ys.inverse_transform(nn_model.predict(xs.transform(X), verbose=0).reshape(-1, 1)).ravel()  # NN prediction in MPa
            out['F'] = np.column_stack([mass, stress])  # both objectives to be minimised

    # NSGA-II setup: LHS for initial spread, SBX crossover and polynomial mutation are pymoo defaults
    alg = NSGA2(pop_size=pop_size, sampling=LHS(), crossover=SBX(prob=0.9, eta=15), mutation=PM(prob=0.2, eta=20), eliminate_duplicates=True)

    # Run optimisation, time it, keep the history for plotting all evaluated points later
    t0  = time.time()  # NSGA-II time
    res = minimize(PlateProblem(), alg, ('n_gen', n_gen), seed=SEED, verbose=False, save_history=True)
    return res, time.time() - t0


print('All functions defined')

## Optimisation 1 - Training Sample Size

In [ ]:
SAMPLE_SIZES = [60, 100, 160, 200, 260, 300]

rows = []
for n in SAMPLE_SIZES:
    # *_ ignores everything except the last three returns (valuable metrics)
    *_, rmse, r2, t = train_and_evaluate_nn(n, N_HIDDEN_DEFAULT, N_NEURONS_DEFAULT, ACTIVATION_DEFAULT, LR_DEFAULT, BATCH_DEFAULT, PATIENCE_DEFAULT)
    # store the run's metrics
    rows.append({'N_train': n, 'RMSE (MPa)': round(rmse, 4), 'R²': round(r2, 4), 'Time (s)': round(t, 2)})
    # prints one line per run so progress is visible while loop runs
    print(f'N={n:>4}  /  RMSE: {rmse:.4f}  /  R²: {r2:.4f}  /  {t:.2f} s')  # >4 left-pads for lineup

df_samples = pd.DataFrame(rows)  # convert the list of dicts into DataFrame
best_idx = df_samples['RMSE (MPa)'].idxmin()  # sample size with lowest RMSE
N_TRAIN_DEFAULT = df_samples.loc[best_idx, 'N_train']  # save it for next runs
print(f'\nSelected Sample Size = {N_TRAIN_DEFAULT}')

## Optimisation 2 - Network Architecture

In [ ]:
ARCHITECTURES = [(1, 8), (2, 8), (1, 16), (2, 16), (1, 32), (2, 32), (1, 64), (2, 64)]

rows = []
for n_h, n_n in ARCHITECTURES:
    label = f'{n_h}x{n_n}'
    *_, rmse, r2, t = train_and_evaluate_nn(N_TRAIN_DEFAULT, n_h, n_n, ACTIVATION_DEFAULT, LR_DEFAULT, BATCH_DEFAULT, PATIENCE_DEFAULT)
    # store N_hidden and N_neurons separately so they can be retrieved directly from candidates
    rows.append({'Architecture': label, 'N_hidden': n_h, 'N_neurons': n_n, 'RMSE (MPa)': round(rmse, 4), 'R²': round(r2, 4), 'Time (s)': round(t, 2)})
    print(f'Arch: {label:<8}  /  RMSE: {rmse:.4f}  /  R²: {r2:.4f}  /  {t:.2f} s')  # <8 right-pads for lineup

df_arch = pd.DataFrame(rows)
best_rmse = df_arch['RMSE (MPa)'].min()
candidates = df_arch[df_arch['RMSE (MPa)'] <= best_rmse * THRESHOLD]  # keeping architectures within 15% of threshold
N_HIDDEN_DEFAULT = int(candidates.loc[candidates['Time (s)'].idxmin(), 'N_hidden'])   # of that list, choosing quickest one
N_NEURONS_DEFAULT = int(candidates.loc[candidates['Time (s)'].idxmin(), 'N_neurons'])
print(f'\nSelected architecture = {N_HIDDEN_DEFAULT}x{N_NEURONS_DEFAULT}')

## Optimisation 3 - Activation Function

In [ ]:
ACTIVATIONS = ['relu', 'tanh', 'sigmoid', 'leaky_relu']

rows = []
for act in ACTIVATIONS:
    *_, rmse, r2, t = train_and_evaluate_nn(N_TRAIN_DEFAULT, N_HIDDEN_DEFAULT, N_NEURONS_DEFAULT, act, LR_DEFAULT, BATCH_DEFAULT, PATIENCE_DEFAULT)
    rows.append({'Activation': act, 'RMSE (MPa)': round(rmse, 4), 'R²': round(r2, 4), 'Time (s)': round(t, 2)})
    print(f'Activation: {act:>10}  /  RMSE: {rmse:.4f}  /  R²: {r2:.4f}  /  {t:.2f} s')  # >10 left-pads for lineup

df_act = pd.DataFrame(rows)
best_rmse = df_act['RMSE (MPa)'].min()
candidates = df_act[df_act['RMSE (MPa)'] <= best_rmse * THRESHOLD]  # keeping activations within 15% of threshold
ACTIVATION_DEFAULT = candidates.loc[candidates['Time (s)'].idxmin(), 'Activation']  # of that list, choosing quickest one
print(f'\nSelected activation = {ACTIVATION_DEFAULT}')

## Optimisation 4 - Learning Rate

In [ ]:
LEARNING_RATES = [0.05, 0.01, 0.005, 0.001, 0.0005, 0.0001]

rows = []
for lr in LEARNING_RATES:
    *_, rmse, r2, t = train_and_evaluate_nn(N_TRAIN_DEFAULT, N_HIDDEN_DEFAULT, N_NEURONS_DEFAULT, ACTIVATION_DEFAULT, lr, BATCH_DEFAULT, PATIENCE_DEFAULT)
    rows.append({'LR': lr, 'RMSE (MPa)': round(rmse, 4), 'R²': round(r2, 4), 'Time (s)': round(t, 2)})
    print(f'LR: {lr:.5f}  /  RMSE: {rmse:.4f}  /  R²: {r2:.4f}  /  {t:.2f} s')

df_lr = pd.DataFrame(rows)
best_rmse = df_lr['RMSE (MPa)'].min()
candidates = df_lr[df_lr['RMSE (MPa)'] <= best_rmse * THRESHOLD]  # keeping learning rates within 15% of threshold
LR_DEFAULT = float(candidates.loc[candidates['Time (s)'].idxmin(), 'LR'])  # of that list, choosing quickest one
print(f'\nSelected LR = {LR_DEFAULT}')

## Optimisation 5 - Batch Size

In [ ]:
BATCH_SIZES = [16, 32, 64]

rows = []
for bs in BATCH_SIZES:
    *_, rmse, r2, t = train_and_evaluate_nn(N_TRAIN_DEFAULT, N_HIDDEN_DEFAULT, N_NEURONS_DEFAULT, ACTIVATION_DEFAULT, LR_DEFAULT, bs, PATIENCE_DEFAULT)
    rows.append({'Batch size': bs, 'RMSE (MPa)': round(rmse, 4), 'R²': round(r2, 4), 'Time (s)': round(t, 2)})
    print(f'Batch: {bs:>4}  /  RMSE: {rmse:.4f}  /  R²: {r2:.4f}  /  {t:.2f} s')  # >4 left-pads for lineup

df_batch = pd.DataFrame(rows)
best_rmse = df_batch['RMSE (MPa)'].min()
candidates = df_batch[df_batch['RMSE (MPa)'] <= best_rmse * THRESHOLD]  # keeping batch sizes within 15% of threshold
BATCH_DEFAULT = int(candidates.loc[candidates['Time (s)'].idxmin(), 'Batch size'])  # of that list, choosing quickest one
print(f'\nSelected batch size = {BATCH_DEFAULT}')

## Optimisation 6 - Early Stopping Patience

In [ ]:
PATIENCE_VALUES = [10, 25, 50]

rows = []
for p in PATIENCE_VALUES:
    *_, rmse, r2, t = train_and_evaluate_nn( N_TRAIN_DEFAULT, N_HIDDEN_DEFAULT, N_NEURONS_DEFAULT, ACTIVATION_DEFAULT, LR_DEFAULT, BATCH_DEFAULT, p)
    rows.append({'Patience': p, 'RMSE (MPa)': round(rmse, 4), 'R²': round(r2, 4), 'Time (s)': round(t, 2)})
    print(f'Patience: {p:>4}  /  RMSE: {rmse:.4f}  /  R²: {r2:.4f}  /  {t:.2f} s')  # >4 left-pads for lineup

df_patience = pd.DataFrame(rows)
best_rmse = df_patience['RMSE (MPa)'].min()
candidates = df_patience[df_patience['RMSE (MPa)'] <= best_rmse * THRESHOLD]  # keeping patience values within 15% of threshold
PATIENCE_DEFAULT = int(candidates.loc[candidates['Time (s)'].idxmin(), 'Patience'])  # of that list, choosing quickest one
print(f'\nSelected patience = {PATIENCE_DEFAULT}')

## Final NN Configuration

In [ ]:
# Final NN with the chosen settings
N_TRAIN_FINAL = N_TRAIN_DEFAULT    # optimised values set as final
N_HIDDEN_FINAL = N_HIDDEN_DEFAULT
N_NEURONS_FINAL = N_NEURONS_DEFAULT
ACTIVATION_FINAL = ACTIVATION_DEFAULT
LR_FINAL = LR_DEFAULT
BATCH_FINAL = BATCH_DEFAULT
PATIENCE_FINAL = PATIENCE_DEFAULT

# Train across 5 seeds - NNs are sensitive to initialisation so multiple runs guard against lucky/unlucky starts
rows = []
weights_store = {}  # keep weights for each seed so the best can be restored without re-training
scalers_store = {}
data_store = {}
epochs_store = {}

for s in range(1, 6):
    (model_s, xs_s, ys_s, X_tr_s, X_val_s, X_test_s, y_tr_s, y_val_s, y_test_s, y_pred_s, n_epochs_s, rmse_s, r2_s, time_s) = train_and_evaluate_nn(N_TRAIN_FINAL, N_HIDDEN_FINAL, N_NEURONS_FINAL, ACTIVATION_FINAL, LR_FINAL, BATCH_FINAL, PATIENCE_FINAL, seed=s)
    rows.append({'Seed': s, 'RMSE (MPa)': round(rmse_s, 4), 'R²': round(r2_s, 4),'Time (s)': round(time_s, 2), 'Epochs': n_epochs_s})
    weights_store[s] = model_s.get_weights()
    scalers_store[s] = (xs_s, ys_s)
    data_store[s] = (X_tr_s, X_val_s, X_test_s, y_tr_s, y_val_s, y_test_s, y_pred_s, rmse_s, r2_s, time_s)
    epochs_store[s] = n_epochs_s
    print(f'Seed {s}  /  RMSE: {rmse_s:.4f}  /  R²: {r2_s:.4f}  /  {time_s:.2f} s  /  Epochs: {n_epochs_s}')

# Pick the best seed using the same 15% threshold + fastest time logic
df_seeds = pd.DataFrame(rows)
best_rmse = df_seeds['RMSE (MPa)'].min()
candidates = df_seeds[df_seeds['RMSE (MPa)'] <= best_rmse * THRESHOLD]  # seeds within 15% of best RMSE
best_seed = int(candidates.loc[candidates['Time (s)'].idxmin(), 'Seed'])  # of that list, choosing quickest one

# Restore the chosen model from saved weights
tf.keras.backend.clear_session()
nn_final = build_nn_model(4, N_HIDDEN_FINAL, N_NEURONS_FINAL, ACTIVATION_FINAL, LR_FINAL)
nn_final.set_weights(weights_store[best_seed])
xs_final, ys_final = scalers_store[best_seed]
(X_tr_f, X_val_f, X_test_f, y_tr_f, y_val_f, y_test_f, y_pred_f, rmse_f, r2_f, time_f) = data_store[best_seed]

# Prints the final solution, training points, and settings used
print(f'\nSelected seed = {best_seed}  /  RMSE: {rmse_f:.4f}  /  R²: {r2_f:.4f}  /  Epochs: {epochs_store[best_seed]}')
print(f'Trained on {len(X_tr_f)} pts  /  Validated on {len(X_val_f)}  /  Tested on {len(X_test_f)}')
print(f'Settings  /  N={N_TRAIN_FINAL}  /  Arch={N_HIDDEN_FINAL}x{N_NEURONS_FINAL}  /  Act={ACTIVATION_FINAL}  /  LR={LR_FINAL}  /  Batch={BATCH_FINAL}  /  Patience={PATIENCE_FINAL}')

# Validation plot: predicted vs actual stress on the test set
lims = [y_test_f.min(), y_test_f.max()]  # axis limits for the ideal line
plt.figure(figsize=(6, 6))
plt.plot(lims, lims, 'k--', label='Ideal')  # y = x reference line
plt.scatter(y_test_f, y_pred_f, alpha=0.8)  # one point per test sample
plt.xlabel('Actual Test Stress (MPa)')
plt.ylabel('Predicted Stress (MPa)')
plt.title('NN Validation Accuracy')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

## NSGA-II Hyperparameter Search

In [ ]:
COMBINATIONS = [(50, 50), (100, 50), (100, 100), (150, 100), (150, 150)]

rows = []
for pop, gen in COMBINATIONS:
    res_i, t_i = run_nsga2(nn_final, xs_final, ys_final, pop, gen)  # run NSGA-II on trained NN
    F_i = res_i.F  # objective values of the Pareto front (mass, stress)
    n_pareto = len(F_i)  # number of points on the front
    F_sorted = F_i[np.argsort(F_i[:, 0])]  # sort the front by mass so points are progressive neighbours
    d = np.linalg.norm(np.diff(F_sorted, axis=0), axis=1)  # Euclidean distance between each pair of consecutive Pareto points
    # mean and std of spacing
    mean_sp = d.mean() if len(d) > 0 else 0.0
    std_sp = d.std()  if len(d) > 0 else 0.0
    # print results for each round
    rows.append({'Pop': pop, 'Gen': gen, 'Pareto pts': n_pareto, 'Mean spacing': round(mean_sp, 4), 'Spacing std': round(std_sp, 4), 'Time (s)': round(t_i, 2)})
    print(f'Pop={pop:>4}   /  Gen={gen:>4}  /  Pareto: {n_pareto:>3}  /  Sp.mean: {mean_sp:.4f}  /  Sp.std: {std_sp:.4f}  |  {t_i:.2f} s')

# Choose the combination within 15% of the lowest Sp.mean + Sp.std, then take the fastest
df_nsga = pd.DataFrame(rows)
df_nsga['Score'] = df_nsga['Mean spacing'] + df_nsga['Spacing std']
best_score = df_nsga['Score'].min()
candidates = df_nsga[df_nsga['Score'] <= best_score * THRESHOLD]
POP_FINAL = int(candidates.loc[candidates['Time (s)'].idxmin(), 'Pop']) # Of that list, choosing quickest one
GEN_FINAL = int(candidates.loc[candidates['Time (s)'].idxmin(), 'Gen'])
print(f'\nSelected Population = {POP_FINAL} and Generations = {GEN_FINAL}')

## Final Run - NN + Pareto Front

In [ ]:
# Getting training + Pareto runtime
# Run NSGA-II on the final NN surrogate (already trained above - no need to re-train)
res_final, time_nsga = run_nsga2(nn_final, xs_final, ys_final, POP_FINAL, GEN_FINAL)

print(f'NN training time  : {time_f:.2f} s')
print(f'NSGA-II time      : {time_nsga:.2f} s')
print(f'Total pipeline    : {time_f + time_nsga:.2f} s')

## Pareto Front

In [ ]:
# Pareto front analysis and plot
# Pull the design variables X and objective values F out of NSGA-II result
X_pareto = res_final.X
F_pareto = res_final.F
mass_pareto = F_pareto[:, 0]
stress_pareto = F_pareto[:, 1]

# Build a DataFrame so Pareto designs are easy to reference in the report
pareto_df = pd.DataFrame({'W1_m': X_pareto[:, 0], 'W2_m': X_pareto[:, 1], 'R_m':  X_pareto[:, 2], 't_m':  X_pareto[:, 3], 'Mass_kg': mass_pareto, 'PredictedStress_MPa': stress_pareto})

def normalise(s): return (s - s.min()) / (s.max() - s.min())

# Normalise both objectives to [0, 1] so mass and stress can be compared
mass_n = normalise(pareto_df['Mass_kg'])
stress_n = normalise(pareto_df['PredictedStress_MPa'])
pareto_df['DistToIdeal'] = np.hypot(mass_n, stress_n)  # Euclidean distance to (0, 0)

# These are the solutions:
lowest_stress = pareto_df['PredictedStress_MPa'].idxmin()  # lowest stress
lowest_mass = pareto_df['Mass_kg'].idxmin()              # lowest mass
optimal = pareto_df['DistToIdeal'].idxmin()          # optimal compromise

# Print the four design variables and both objectives for each notable design
cols = ['W1_m', 'W2_m', 'R_m', 't_m', 'Mass_kg', 'PredictedStress_MPa']
print('Lowest stress point:'); print(pareto_df.loc[lowest_stress, cols])
print('\nLowest mass point:'); print(pareto_df.loc[lowest_mass, cols])
print('\nOptimal compromise:'); print(pareto_df.loc[optimal, cols])

# Every solution NSGA-II ever evaluated across all generations
all_sols = np.vstack([g.pop.get('F') for g in res_final.history])
plt.figure(figsize=(7, 7))
plt.scatter(all_sols[:, 0], all_sols[:, 1], c='gray', alpha=0.25, label='All solutions')  # all solutions (non-Pareto)
plt.scatter(mass_pareto, stress_pareto, c='blue', label='Pareto front')   # Pareto solutions
plt.scatter(pareto_df.loc[lowest_stress, 'Mass_kg'], pareto_df.loc[lowest_stress,'PredictedStress_MPa'], color='red',    s=120, zorder=5, label='Lowest stress')
plt.scatter(pareto_df.loc[lowest_mass, 'Mass_kg'], pareto_df.loc[lowest_mass, 'PredictedStress_MPa'], color='green',  s=120, zorder=5, label='Lowest mass')
plt.scatter(pareto_df.loc[optimal, 'Mass_kg'], pareto_df.loc[optimal, 'PredictedStress_MPa'], color='orange', s=120, zorder=5, label='Optimal compromise')
plt.xlabel('Mass (kg)')
plt.ylabel('Stress (MPa)')
plt.title('Pareto Front - NN + NSGA-II')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# End of clock
notebook_elapsed = time.time() - notebook_start
print(f'Total wall-clock time: {notebook_elapsed:.1f} s')